# 07 — Model Explainability (SHAP)

Explains the best model from `06_cancellation_prediction.ipynb` with
SHAP, answering: **which features drive cancellation-risk predictions,
and how?** Rebuilds the same time-based split + encoding used in
notebook 06 (feature engineering itself lives in notebook 05 — this
notebook only re-derives the train/test matrices so it can run
independently), loads whichever model file training actually produced,
and explains it directly rather than assuming XGBoost was available.

In [1]:
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
IMAGES_DIR = Path("../images/ml")

/Users/navdeeptaliyan/Downloads/files/Ride-Sharing-Analytics-Cancellation-Prediction/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the model comparison results and pick the winning model file

In [2]:
model_comparison = pd.read_csv(PROCESSED_DIR / "model_comparison.csv")
best_row = model_comparison.sort_values("f1_score", ascending=False).iloc[0]
best_model_name = best_row["model"]
print(f"Best model per model_comparison.csv: {best_model_name}")

model_filename = "cancellation_xgboost.pkl" if best_model_name == "XGBoost" else \
    f"cancellation_{best_model_name.lower().replace(' ', '_')}.pkl"
best_model = joblib.load(MODELS_DIR / model_filename)
print(f"Loaded models/{model_filename}")

Best model per model_comparison.csv: Random Forest
Loaded models/cancellation_random_forest.pkl


## 2. Rebuild the exact train/test matrices used to train it

In [3]:
df = pd.read_csv(PROCESSED_DIR / "trips_ml_ready.csv", parse_dates=["request_datetime"])
df = df.sort_values("request_datetime").reset_index(drop=True)

ID_COLS = ["trip_id", "rider_id", "driver_id", "request_datetime"]
TARGET = "is_cancelled"
CATEGORICAL = ["vehicle_type", "pickup_city", "drop_city", "payment_method",
               "rider_gender", "preferred_payment"]

combined = pd.get_dummies(df.drop(columns=ID_COLS + [TARGET]), columns=CATEGORICAL, drop_first=True)
feature_names = combined.columns.tolist()

split_idx = int(len(df) * 0.8)
X_train = combined.iloc[:split_idx].reset_index(drop=True)
X_test = combined.iloc[split_idx:].reset_index(drop=True)

is_linear_model = isinstance(best_model, LogisticRegression)
if is_linear_model:
    scaler = StandardScaler()
    X_train_model = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names)
    X_test_model = pd.DataFrame(scaler.transform(X_test), columns=feature_names)
else:
    X_train_model, X_test_model = X_train, X_test

# SHAP on a sample of the test set for speed — swap for X_test_model to explain everything
SAMPLE_SIZE = min(1500, len(X_test_model))
X_explain = X_test_model.sample(SAMPLE_SIZE, random_state=42)
print(f"Explaining {SAMPLE_SIZE} test-set predictions from {best_model_name}")

Explaining 1500 test-set predictions from Random Forest


## 3. Build the right SHAP explainer for the model type

Tree models (Random Forest, XGBoost) get the fast, exact `TreeExplainer`.
Logistic Regression gets `LinearExplainer`, which accounts for the
StandardScaler-transformed feature space it was trained on.

In [4]:
if isinstance(best_model, RandomForestClassifier):
    explainer = shap.TreeExplainer(best_model)
    shap_values_raw = explainer.shap_values(X_explain)
    # sklearn RF binary classifiers: TreeExplainer returns a list [class0, class1] or a 3-D array
    if isinstance(shap_values_raw, list):
        shap_values = shap_values_raw[1]
    elif shap_values_raw.ndim == 3:
        shap_values = shap_values_raw[:, :, 1]
    else:
        shap_values = shap_values_raw
elif is_linear_model:
    explainer = shap.LinearExplainer(best_model, X_train_model)
    shap_values = explainer.shap_values(X_explain)
else:  # XGBoost
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_explain)

shap_values = np.array(shap_values)
print(f"SHAP values shape: {shap_values.shape}")

SHAP values shape: (1500, 42)


## 4. Global feature importance

In [5]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feature_importance = (
    pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
feature_importance.to_csv(PROCESSED_DIR / "shap_feature_importance.csv", index=False)
print(feature_importance.head(15))

                         feature  mean_abs_shap
0               surge_multiplier       0.028670
1                    distance_km       0.023382
2              driver_avg_rating       0.014694
3       driver_prior_cancel_rate       0.013465
4                   is_peak_hour       0.012994
5         driver_experience_days       0.012092
6                   request_hour       0.008039
7   driver_prior_completed_count       0.003854
8                    request_dow       0.002090
9        driver_prior_trip_count       0.001957
10                    is_weekend       0.001678
11   days_since_driver_last_trip       0.001652
12             vehicle_type_Bike       0.001218
13                     base_fare       0.001130
14    days_since_rider_last_trip       0.001118


## 5. SHAP summary plots

In [6]:
plt.figure()
shap.summary_plot(shap_values, X_explain, plot_type="bar", show=False, max_display=15)
plt.title(f"Global Feature Importance — {best_model_name}")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "03_shap_bar.png", dpi=140, bbox_inches="tight")
plt.close()

plt.figure()
shap.summary_plot(shap_values, X_explain, show=False, max_display=15)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "04_shap_summary.png", dpi=140, bbox_inches="tight")
plt.close()

## 6. Individual prediction explanation

Explains a single high-risk trip — the kind of "why did the model flag
this one?" answer that makes the model usable by ops/business teams.

In [7]:
row_idx = int(np.argmax(shap_values.sum(axis=1)))  # most "pushed toward cancellation" example
plt.figure()
shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value if not isinstance(explainer.expected_value, np.ndarray)
    else explainer.expected_value[-1],
    shap_values[row_idx],
    feature_names=feature_names,
    max_display=12,
    show=False,
)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "05_shap_waterfall_example.png", dpi=140, bbox_inches="tight")
plt.close()

print(f"\nSHAP explainability complete for {best_model_name}.")
print("Saved: data/processed/shap_feature_importance.csv, "
      "images/ml/03_shap_bar.png, 04_shap_summary.png, 05_shap_waterfall_example.png")


SHAP explainability complete for Random Forest.
Saved: data/processed/shap_feature_importance.csv, images/ml/03_shap_bar.png, 04_shap_summary.png, 05_shap_waterfall_example.png
